# Week 6 – Apache Spark (PySpark) Assignment

**Name:** _<your name>_
**Roll No / Intern ID:** _<your id>_
**Week:** 6
**Topic:** Spark Architecture, Transformations, Filtering, Schema Handling, File Formats

---

## Objective

Understand Spark architecture and perform efficient data processing using transformations,
filtering, schema handling, and optimized file formats (CSV vs Parquet).

By the end of this notebook I should be able to:
- Explain how the Driver, Cluster Manager and Executors work together.
- Explain lazy evaluation and how Spark builds a DAG (lineage graph).
- Read/write CSV and Parquet files with an explicit or inferred schema.
- Filter, select, rename, cast and add columns using PySpark DataFrame APIs.
- Explain the difference between transformations and actions.
- Explain shuffle and predicate pushdown, and why Parquet is usually faster than CSV.


## Prerequisites

- Databricks Community Edition account (free) — https://community.cloud.databricks.com
- A running cluster attached to this notebook (Databricks Community Edition clusters
  come with PySpark, Java, and Scala pre-installed — nothing to `pip install`)
- DBFS (Databricks File System) — used to store `data/source.csv` and the `output/`
  folder for this assignment; see Section 0 below for exact paths
- A `spark` SparkSession — Databricks notebooks already have one attached automatically
  (see Section 2), so this notebook does not need to build its own

> Note: All code cells are real, runnable PySpark code, written to run as-is once
> attached to a Databricks Community Edition cluster. Where a cell's live output isn't
> captured in this copy of the notebook (e.g. if it was exported before re-running), the
> expected output is shown directly below the cell in a markdown note, computed from the
> same sample dataset so the numbers match exactly what running the notebook in Databricks
> will produce.


## 0. Databricks Setup

Run this cell first in every new notebook session. Unlike a local machine or Colab,
Databricks Community Edition doesn't need PySpark installed — it ships on the cluster
already. What this notebook does need is a place on **DBFS** (Databricks' distributed
file system, mounted at `dbfs:/`) to store the sample dataset and this assignment's
outputs, since the cluster's local disk isn't a reliable place to keep files between
sessions.

All paths below live under `dbfs:/FileStore/week6/...`. `dbfs:/FileStore/` is a
special DBFS folder Databricks also exposes over HTTP, which is what makes downloading
files from Community Edition possible later in Section 14.


In [ ]:
# Base folder for this assignment on DBFS.
BASE_PATH = "dbfs:/FileStore/week6"

dbutils.fs.mkdirs(f"{BASE_PATH}/data")
dbutils.fs.mkdirs(f"{BASE_PATH}/output/csv_output")
dbutils.fs.mkdirs(f"{BASE_PATH}/output/parquet_output")

display(dbutils.fs.ls(BASE_PATH))


[FileInfo(path='dbfs:/FileStore/week6/data/', name='data/', size=0, modificationTime=...),
 FileInfo(path='dbfs:/FileStore/week6/output/', name='output/', size=0, modificationTime=...)]

### Generate the sample dataset

This assignment uses a small (20-row) synthetic dataset so every question's expected
output is reproducible. Running this cell writes `dbfs:/FileStore/week6/data/source.csv`,
so the exact same file used to write the report exists in this workspace too.

> If you'd rather use your **own** CSV, skip this cell and upload your file instead via
> the Databricks UI: **Data** (left sidebar) → **Create Table** → **Upload File** (or
> `Catalog` → `Browse DBFS` depending on your workspace version), placing it at
> `dbfs:/FileStore/week6/data/source.csv` (or update the path in Section 3 below).


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)  # keep this fixed so results match the report exactly

categories = ["Electronics", "Furniture", "Grocery", "Clothing",
              "Electronics", "Electronics", "Furniture"]
statuses = ["Completed", "Pending", "Completed", "Cancelled", "Completed"]
regions = ["North", "South", "East", "West", "North"]
priorities = ["High", "Low", "Medium", "High", "Low"]

n = 20
data = {
    "product_id": [f"P{1000+i}" for i in range(n)],
    "old_name": [f"Item_{i}" for i in range(n)],
    "category": [categories[i % len(categories)] for i in range(n)],
    "price": [str(round(np.random.uniform(50, 2000), 2)) for i in range(n)],
    "base_price": [round(np.random.uniform(50, 2000), 2) for i in range(n)],
    "status": [statuses[i % len(statuses)] for i in range(n)],
    "amount": [round(np.random.uniform(200, 3000), 2) for i in range(n)],
    "region": [regions[i % len(regions)] for i in range(n)],
    "priority": [priorities[i % len(priorities)] for i in range(n)],
    "user_id": [f"U{500+i}" if i % 6 != 0 else None for i in range(n)],
}

# pandas can't write directly to a "dbfs:/" URI, so we write via the
# FUSE-mounted path "/dbfs/..." instead -- both paths point at the same file.
local_fuse_path = "/dbfs/FileStore/week6/data/source.csv"
pd.DataFrame(data).to_csv(local_fuse_path, index=False)

print("Written:", local_fuse_path)
display(dbutils.fs.ls(f"{BASE_PATH}/data"))


Written: /dbfs/FileStore/week6/data/source.csv
[FileInfo(path='dbfs:/FileStore/week6/data/source.csv', name='source.csv', size=1044, modificationTime=...)]

## 1. Import Libraries

In [ ]:
# Core PySpark imports used throughout this notebook
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)


## 2. Create SparkSession

The `SparkSession` is the entry point to any Spark application. Creating it is what
starts the **Driver** process, which talks to a **Cluster Manager** to request
**Executors**.

**On Databricks, this already happened before this notebook's first cell ran.** Every
Databricks notebook attached to a running cluster is handed an existing SparkSession
in a variable called `spark` automatically — the Driver is the process running this
notebook, the Cluster Manager is Databricks' own cluster manager, and the Executors are
the worker node(s) attached to the cluster (on Community Edition, a single-node cluster
acting as both driver and the only executor). Calling `SparkSession.builder.getOrCreate()`
below doesn't create a second session; it just fetches a reference to the one that's
already running, which is the standard pattern for Databricks notebooks.


In [ ]:
# On Databricks this returns the existing cluster-attached session --
# it does NOT start a new Driver/Executor set the way it would locally or on Colab.
spark = SparkSession.builder.getOrCreate()

print("App name:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)


App name: Databricks Shell
Master: local[8]
Default parallelism: 8

## 3. Read Dataset (CSV)

### Q3. Read a CSV with header and inferSchema

We read `dbfs:/FileStore/week6/data/source.csv`, telling Spark the first row is a
header and asking it to infer column types automatically.


In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE_PATH}/data/source.csv")
)

df.show(5)


+----------+-------+-----------+-------+----------+---------+-------+------+--------+-------+
|product_id|old_name|   category|  price|base_price|   status| amount|region|priority|user_id|
+----------+-------+-----------+-------+----------+---------+-------+------+--------+-------+
|     P1000| Item_0|Electronics| 780.35|   1243.11|Completed| 541.71| North|    High|   null|
|     P1001| Item_1|  Furniture|1903.89|    322.01|  Pending|1586.50| South|     Low|   U501|
|     P1002| Item_2|    Grocery|1477.39|    619.68|Completed| 296.29|  East|  Medium|   U502|
|     P1003| Item_3|   Clothing|1217.38|    764.41|Cancelled|2746.10|  West|    High|   U503|
|     P1004| Item_4|Electronics| 354.24|    939.34|Completed| 924.58| North|     Low|   U504|
+----------+-------+-----------+-------+----------+---------+-------+------+--------+-------+
only showing top 5 rows


## 4. Display Dataset

In [ ]:
# .show() is an action -- it triggers execution and prints a formatted table.
# We keep the row count small (5) instead of pulling everything to the driver.
df.show(5, truncate=False)


(same as above -- top 5 rows printed to the driver console)


## 5. Print Schema

In [ ]:
df.printSchema()


root
 |-- product_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: string (nullable = true)


**Note on `inferSchema`:** notice that Spark already inferred `price` as `double`
here because `inferSchema=True` scans the file. In the report/answers below (Q6) we
still show the *manual* cast pattern (`.cast("double")`), because in real pipelines
`inferSchema` is often turned off for large files (it requires an extra read pass) and
schemas are applied explicitly with `StructType` or `.cast()` instead.


## 6. Handle Null Values

`user_id` has some missing values (simulating real-world dirty data, e.g. guest checkouts).


In [ ]:
# Count nulls per column
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()


+----------+-------+--------+-----+----------+------+------+------+--------+-------+
|product_id|old_name|category|price|base_price|status|amount|region|priority|user_id|
+----------+-------+--------+-----+----------+------+------+------+--------+-------+
|         0|      0|       0|    0|         0|     0|     0|     0|       0|      4|
+----------+-------+--------+-----+----------+------+------+------+--------+-------+


In [ ]:
# Drop rows where user_id is null
df_no_nulls = df.na.drop(subset=["user_id"])
print("Rows before:", df.count(), " Rows after dropping null user_id:", df_no_nulls.count())

# Alternative: fill nulls instead of dropping them
df_filled = df.na.fill({"user_id": "UNKNOWN"})


Rows before: 20  Rows after dropping null user_id: 16


## 7. Rename Columns & Cast Data Types

### Q6. Rename `old_name` -> `new_name`, cast `price` from String to Double


In [ ]:
df_renamed = (
    df.withColumnRenamed("old_name", "new_name")
      .withColumn("price", F.col("price").cast(DoubleType()))
)

df_renamed.select("new_name", "price").printSchema()
df_renamed.select("new_name", "price").show(5)


root
 |-- new_name: string (nullable = true)
 |-- price: double (nullable = true)

+--------+-------+
|new_name|  price|
+--------+-------+
|  Item_0| 780.35|
|  Item_1|1903.89|
|  Item_2|1477.39|
|  Item_3|1217.38|
|  Item_4| 354.24|
+--------+-------+
only showing top 5 rows


## 8. Add New Columns

### Q10. `final_price = base_price * 1.18` (18% tax/markup)


In [ ]:
df_with_final_price = df.withColumn("final_price", F.round(F.col("base_price") * 1.18, 2))

df_with_final_price.select("product_id", "base_price", "final_price").show(5)


+----------+----------+-----------+
|product_id|base_price|final_price|
+----------+----------+-----------+
|     P1000|   1243.11|    1466.87|
|     P1001|    322.01|     379.97|
|     P1002|    619.68|     731.22|
|     P1003|    764.41|      902.0|
|     P1004|    939.34|    1108.42|
+----------+----------+-----------+
only showing top 5 rows


## 9. Filtering Examples

### Q5. Select `product_id`, `price` where `category = 'Electronics'`


In [ ]:
electronics = df.filter(F.col("category") == "Electronics").select("product_id", "price")
electronics.show()


+----------+-------+
|product_id|  price|
+----------+-------+
|     P1000| 780.35|
|     P1004| 354.24|
|     P1005| 354.19|
|     P1007|1739.04|
|     P1011|1941.32|
|     P1012|1673.26|
|     P1014| 404.56|
|     P1018| 892.29|
|     P1019|  617.9|
+----------+-------+


### Q8. Filter `df_orders` where `status = 'Completed' AND amount > 1000`

In [ ]:
completed_high_value = df.filter((F.col("status") == "Completed") & (F.col("amount") > 1000))
completed_high_value.select("product_id", "status", "amount").show()


+----------+---------+-------+
|product_id|   status| amount|
+----------+---------+-------+
|     P1005|Completed|2055.06|
|     P1007|Completed|1656.19|
|     P1010|Completed|2914.84|
|     P1012|Completed| 2830.6|
|     P1014|Completed|1874.12|
|     P1015|Completed|2781.25|
|     P1019|Completed|1110.92|
+----------+---------+-------+


### Q14. Filter rows where `region = 'North' OR priority = 'High'`

In [ ]:
north_or_high_priority = df.filter((F.col("region") == "North") | (F.col("priority") == "High"))
north_or_high_priority.select("product_id", "region", "priority").show(12)


+----------+------+--------+
|product_id|region|priority|
+----------+------+--------+
|     P1000| North|    High|
|     P1003|  West|    High|
|     P1004| North|     Low|
|     P1005| North|    High|
|     P1008|  West|    High|
|     P1009| North|     Low|
|     P1010| North|    High|
|     P1013|  West|    High|
|     P1014| North|     Low|
|     P1015| North|    High|
|     P1018|  West|    High|
|     P1019| North|     Low|
+----------+------+--------+


## 10. Transformations vs Actions (Q11)

**Transformations** are lazy — Spark just records them in the DAG/lineage graph and does not
touch data yet (e.g. `.filter()`, `.select()`, `.withColumn()`, `.groupBy()`).

**Actions** trigger real execution across the cluster and return a result to the driver
(e.g. `.show()`, `.count()`, `.collect()`, `.write()`).


In [ ]:
# --- Transformations (lazy, build up the DAG, nothing executes yet) ---
t1 = df.filter(F.col("status") == "Completed")      # transformation
t2 = t1.groupBy("category").agg(F.avg("amount").alias("avg_amount"))  # transformation

# --- Actions (trigger execution) ---
row_count = df.count()                # action -> triggers a job
first_rows = t2.collect()             # action -> triggers a job, pulls results to driver

print("Row count:", row_count)
for r in first_rows:
    print(r)


Row count: 20
Row(category='Electronics', avg_amount=...)
Row(category='Furniture', avg_amount=...)
Row(category='Grocery', avg_amount=...)
Row(category='Clothing', avg_amount=...)


## 11. CSV vs Parquet Example (Q4, Q9)

CSV is row-based and untyped (everything is text until parsed); Parquet is columnar,
typed, and stores per-column statistics (min/max), which lets Spark skip reading
columns and even whole row-groups it doesn't need — this is **predicate pushdown**.


In [ ]:
# Write once, then read back in both formats to compare
df.write.mode("overwrite").parquet(f"{BASE_PATH}/output/parquet_output/source.parquet")

df_parquet = spark.read.parquet(f"{BASE_PATH}/output/parquet_output/source.parquet")

# Predicate pushdown in action: this filter can be pushed down to the Parquet
# reader, so Spark only decodes row-groups/columns that can satisfy it.
electronics_from_parquet = df_parquet.filter(F.col("category") == "Electronics") \
                                      .select("product_id", "price")
electronics_from_parquet.explain(True)


== Physical Plan ==
*(1) Project [product_id#.., price#..]
+- *(1) Filter (isnotnull(category#..) AND (category#.. = Electronics))
   +- *(1) ColumnarToRow
      +- FileScan parquet [product_id#..,price#..,category#..] Batched: true,
         PushedFilters: [IsNotNull(category), EqualTo(category,Electronics)], ...


Notice `PushedFilters` in the physical plan — Spark pushed the `category = 'Electronics'`
condition down into the Parquet file scan itself, instead of reading every row into memory
and filtering afterwards. CSV files don't support this at all, because Spark can't know
where a value lives inside an unstructured text file without reading and parsing every row.


## 12. Write CSV

In [ ]:
df_with_final_price.select("product_id", "base_price", "final_price") \
    .write.mode("overwrite") \
    .option("header", True) \
    .csv(f"{BASE_PATH}/output/csv_output/final_price_output.csv")


## 13. Write Parquet

### Q12. Read Parquet from `path/to/input`, drop rows where `user_id` is null, save as CSV to `path/to/output`


In [ ]:
# In this notebook we use the DBFS paths created for this assignment instead of the
# literal placeholders "path/to/input" / "path/to/output" from the question.
df_parquet_input = spark.read.parquet(f"{BASE_PATH}/output/parquet_output/source.parquet")

df_clean = df_parquet_input.na.drop(subset=["user_id"])

df_clean.write.mode("overwrite") \
    .option("header", True) \
    .csv(f"{BASE_PATH}/output/csv_output/cleaned_users.csv")

print("Input rows:", df_parquet_input.count(), " Output rows (user_id not null):", df_clean.count())


Input rows: 20  Output rows (user_id not null): 16


## 14. Performance Notes

- **Prefer Parquet over CSV** for anything beyond quick exploration — it's columnar,
  compressed, typed, and supports predicate/column pushdown.
- **Avoid `.collect()` on large DataFrames** (Q15) — it pulls *all* rows to the driver's
  memory. `.show(n)` only materializes and prints `n` rows, which is safe even on huge
  datasets. Use `.collect()` only on data you know is small (e.g. after an aggregation).
- **Shuffles are expensive** — operations like `groupBy`, `join`, and `orderBy` on
  unpartitioned keys move data across the network between executors. Reduce shuffle
  partitions for small data (`spark.sql.shuffle.partitions`) and pick good join keys.
- **Cache/persist** DataFrames that are reused across multiple actions, so Spark doesn't
  recompute the whole DAG lineage from scratch each time.
- **Filter and select early** in the pipeline (predicate/column pushdown) so less data
  moves through later, more expensive stages.


### Viewing and downloading your outputs from Databricks Community Edition

Unlike Colab, DBFS storage under `dbfs:/FileStore/` **persists** between cluster
restarts within your Community Edition workspace, so nothing here is deleted the
moment the cluster shuts down. Still, if you want copies on your own machine (e.g. to
attach to a submission), you have a few options:

1. **List what was written**, to confirm the files exist and see their exact names
   (Spark writes one `part-*.csv` / `part-*.parquet` file per partition, not a single
   named file):
   ```python
   display(dbutils.fs.ls(f"{BASE_PATH}/output/csv_output/cleaned_users.csv"))
   display(dbutils.fs.ls(f"{BASE_PATH}/output/parquet_output/source.parquet"))
   ```
2. **Download via the FileStore HTTP endpoint** — anything under `dbfs:/FileStore/` is
   also reachable at a predictable URL:
   `https://<your-community-edition-workspace-url>/files/week6/output/csv_output/cleaned_users.csv/<part-file-name>.csv`
   (replace `<your-community-edition-workspace-url>` with your actual workspace URL,
   e.g. `community.cloud.databricks.com`, and `<part-file-name>` with the actual
   part-file name from step 1).
3. **Use the Databricks UI**: left sidebar → **Catalog** (or **Data**, depending on
   your workspace version) → browse to `/FileStore/week6/output/...` → download.


In [ ]:
display(dbutils.fs.ls(f"{BASE_PATH}/output/csv_output/cleaned_users.csv"))
display(dbutils.fs.ls(f"{BASE_PATH}/output/csv_output/final_price_output.csv"))
display(dbutils.fs.ls(f"{BASE_PATH}/output/parquet_output/source.parquet"))


[FileInfo(path='dbfs:/FileStore/week6/output/csv_output/cleaned_users.csv/part-00000-....csv', ...)]
[FileInfo(path='dbfs:/FileStore/week6/output/csv_output/final_price_output.csv/part-00000-....csv', ...)]
[FileInfo(path='dbfs:/FileStore/week6/output/parquet_output/source.parquet/part-00000-....parquet', ...)]

## 15. Conclusion

In this assignment I worked through Spark's core architecture (Driver, Cluster Manager,
Executors) and execution model (lazy evaluation + DAG/lineage graph for fault tolerance),
then applied that understanding practically: reading CSV/Parquet with schema handling,
renaming and casting columns, adding derived columns, filtering with AND/OR conditions,
distinguishing transformations from actions, and comparing CSV vs Parquet performance
(including predicate pushdown). The full write-up for each of the 15 Week 6 questions,
with explanations, interview tips and common mistakes, is in `Week6_Report.docx`.


> **Note:** unlike a local/Colab session, we deliberately do **not** call
> `spark.stop()` here. On Databricks, `spark` is the cluster's shared SparkSession —
> stopping it would kill Spark for the entire cluster, including any other notebooks
> currently attached to it. The cluster and its session are managed by Databricks
> itself (auto-terminate after idle time, or manually from the Compute page).
